# Self-supervised learned operators vs analytic kernels

Extension of `ssl_analysis.ipynb` (that file is untouched — the metric-pipeline and plotting cells are copied
verbatim below so this notebook is standalone). Encoders trained with `SpectralContrastiveLoss` are treated
**exactly like kernels**: the learned feature map φ induces the kernel k(x,y) = ⟨φ(x), φ(y)⟩, so its n×n Gram
matrix feeds the *unchanged* metric pipeline (`metric_distortion`, `spectral_bias`, `spurious_residual`,
`analyse_spectrum`), and the outputs land in the same CSV schema as `benchmarks/` — directly consumable by
`kernel_spectral_score` and the unified scoring database.

**Systems** (one reversible, one strongly non-normal, one linear, one graph):

| system | kinds | reference spectrum | input dim |
|---|---|---|---|
| Langevin (Prinz) | all | `compute_prinz_potential_eig` | 1 |
| Duffing | ω ∈ {0.5, 1.5, 3.0} | none (data-driven diagnostics only) | 2 |
| Linear | A ∈ {rev, rot} | eig(expm(A·lag)) + products | 2 |
| Graph random walk | cycle_12, path_12 | eig(P) | 12 (one-hot) |

**Sweep:** latent dim d ∈ {4, 8, 16} × 3 seeds; both Ridge variants (Principal Components (kDMD) / RRR) fitted on the same latents.
**Hypotheses:** H1 unified score ranks encoders comparably to the best analytic kernels on the reversible
system; H2 encoders reduce spurious spectrum in non-normal regimes; H3 small d ⇒ representational mismatch
shows up as high distortion η̂ (the score should detect it).

Outputs → `../analysis/ssl/`. Approx. runtime: 72 encoder trainings ≈ 20–40 min CPU + metrics.

## 1. Metric pipeline (copied verbatim from `ssl_analysis.ipynb`)

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import expm

from kooplearn._linalg import eigh_rank_reveal, spd_neg_pow, weighted_norm
from kooplearn.datasets import compute_prinz_potential_eig, make_prinz_potential


def operator_norm_error(true_operator: np.ndarray, estimated_operator: np.ndarray):
    r"""Operator norm error proxy for a Koopman estimator.

    Computes the operator norm discrepancy between the true action
    :math:`A_\pi S` and the estimated action :math:`S \widehat{G}`:

    .. math::

        \mathcal{E}(\widehat{G}) := \|A_\pi S - S \widehat{G}\|.

    Since "kooplearn" does not currently expose :math:`A_\pi` or the embedding
    operator :math:`S` explicitly, this function works with their actions on a
    common finite-dimensional representation. In practice, the caller should pass
    matrices or vectors representing the two quantities to be compared.
    """
    true_operator = np.asanyarray(true_operator)
    estimated_operator = np.asanyarray(estimated_operator)

    if true_operator.shape != estimated_operator.shape:
        raise ValueError(
            "true_operator and estimated_operator must have the same "
            f"shape, got {true_operator.shape} and "
            f"{estimated_operator.shape}."
        )

    diff = true_operator - estimated_operator
    if diff.ndim == 1:
        return float(np.linalg.norm(diff))
    return float(np.linalg.norm(diff, ord=2))


def metric_distortion(psi, C):
    r"""Empirical metric distortion :math:`\widehat\eta_i = \|\widehat\psi_i\|_{\mathcal H} /
    \sqrt{\langle \widehat C \widehat\psi_i, \widehat\psi_i\rangle}`.

    Parameters
    ----------
    psi : ndarray, shape (n,) or (n, k)
        Eigenfunction(s) evaluated at the *training* points. If 2D, each
        column is treated as a separate eigenfunction (see `weighted_norm`).
    C : ndarray, shape (n, n)
        Empirical (kernel-based) covariance, i.e. ``model.kernel_X / n_samples``.
    """
    psi = np.asarray(psi)
    n = C.shape[0]

    # ||psi||_H via the reproducing property: needs the *inverse* Gram, since
    # C = K_X / n is the Gram-based covariance, not the RKHS metric itself.
    C_inv = spd_neg_pow(C * n, exponent=-1.0)  # i.e. K_X^{-1}
    rkhs_norm = weighted_norm(psi, M=C_inv)

    # <C psi, psi> = (1/n)||psi(X)||_2^2, i.e. weighted_norm with M=None, squared, over n
    empirical_norm = weighted_norm(psi, M=None) / np.sqrt(n)

    with np.errstate(divide="ignore", invalid="ignore"):
        eta = rkhs_norm / empirical_norm
    eta = np.where(empirical_norm > 0, eta, np.nan)
    return eta if psi.ndim == 2 else float(eta)


def spectral_bias(eigenfunction, C, rho):
    r"""Empirical spectral bias :math:`\hat s_i = \widehat\eta_i \, \rho_{r+1}`."""
    eta = metric_distortion(eigenfunction, C)
    s_hat = eta * rho
    return float(s_hat), eta


def _top_sv(C, r):
    """(r+1)-st eigenvalue of a symmetric PSD matrix, via eigh_rank_reveal."""
    raw_vals, raw_vecs = np.linalg.eigh(np.asarray(C))
    _, top_vals, _ = eigh_rank_reveal(raw_vals, raw_vecs, rank=r + 1)
    if len(top_vals) <= r:
        return 0.0
    return float(top_vals[-1])


# --- truncation helpers ---
def pcr_truncation(C, r):
    r""":math:`\rho_{r+1}(\widehat G^{Principal Components (kDMD)}) = \sigma_{r+1}(\widehat C)`."""
    return _top_sv(C, r)


# kDMD uses the same (r+1)-st eigenvalue of the empirical covariance as Principal Components (kDMD)
kdmd_truncation = pcr_truncation


def rrr_truncation(C, T, r, cutoff=None):
    r""":math:`\rho_{r+1}(\widehat G^{RRR}) = \sigma_{r+1}(\widehat C^{-1/2}\widehat T)`."""
    C_inv_sqrt = spd_neg_pow(np.asarray(C), exponent=-0.5, cutoff=cutoff)
    A = C_inv_sqrt @ np.asarray(T)
    svals = np.linalg.svd(A, compute_uv=False)
    if r >= len(svals):
        return 0.0
    return float(svals[r])


# --- spectral gap (top-two magnitude eigenvalues) ---
def spectral_gap(eigenvalues):
    mags = np.sort(np.abs(eigenvalues))[::-1]
    return float(mags[0] - mags[1]) if len(mags) > 1 else np.nan


# --- spurious eigenvalues vs reference ---


def spurious_ref(est, ref, delta):
    dist = np.abs(est[:, None] - ref[None, :])
    return int(np.sum(dist.min(axis=1) > delta))


def spurious_residual(eigenvalues, psi_X_val, psi_Y_val, delta, relative=True):
    r"""Data-driven spurious-eigenpair check (see paper Appendix C, Remark 4).

    Flags eigenpairs that fail the empirical consistency check
    :math:`\hat\psi_i(y_j) \approx \hat\lambda_i \hat\psi_i(x_j)` on a
    held-out validation set.

    Parameters
    ----------
    eigenvalues : ndarray, shape (r,)
        Estimated eigenvalues, same order as columns of psi_X_val/psi_Y_val.
    psi_X_val : ndarray, shape (n_val, r)
        Eigenfunctions evaluated at validation inputs x_j.
    psi_Y_val : ndarray, shape (n_val, r)
        Same eigenfunctions evaluated at outputs y_j.
    delta : float
        Threshold on the residual score.
    relative : bool
        If True, normalize residual by ||psi_X_val||.

    Returns
    -------
    n_spurious : int
    scores : ndarray, shape (r,)
    """
    eigenvalues = np.asarray(eigenvalues)
    n_val = psi_X_val.shape[0]

    resid = psi_Y_val - psi_X_val * eigenvalues[None, :]
    resid_norm = weighted_norm(resid) / np.sqrt(n_val)

    if relative:
        base_norm = weighted_norm(psi_X_val) / np.sqrt(n_val)
        scores = np.full_like(resid_norm, np.nan, dtype=float)

        ok = np.isfinite(base_norm) & (base_norm > 0)
        scores[ok] = resid_norm[ok] / base_norm[ok]
    else:
        scores = resid_norm

    n_spurious = int(np.sum(scores > delta))
    return n_spurious, scores


# --- compilation function for analysing spectral metrics ---
def analyse_spectrum(modes_records, trials_records, out_prefix):
    modes_df = pd.DataFrame(modes_records).copy()
    trials_df = pd.DataFrame(trials_records).copy()

    if "spectral_gap" not in modes_df.columns:
        raise ValueError(
            f"modes_df is missing 'spectral_gap'. Columns: {modes_df.columns.tolist()}"
        )

    summary = modes_df.groupby(
        ["kernel", "kind", "method", "eigenfunction_id"], as_index=False
    ).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        dist_mean=("metric_distortion", "mean"),
        trunc_mean=("truncation", "mean"),
        spurious_mean=("residual_spurious_score", "mean"),
        spurious_std=("residual_spurious_score", "std"),
    )

    rows = []
    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        gg = g[["spectral_bias", "spectral_gap"]].dropna()
        corr = gg["spectral_bias"].corr(gg["spectral_gap"]) if len(gg) > 1 else np.nan
        rows.append(
            {
                "kernel": kernel,
                "kind": kind,
                "method": method,
                "bias_gap_corr": corr,
            }
        )
    corr_df = pd.DataFrame(rows)

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    modes_df.to_csv(f"{out_prefix}_metrics.csv", index=False)
    trials_df.to_csv(f"{out_prefix}_trials.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_corr.csv", index=False)

    # ------------------------------------------------------------
    # Scatter diagnostics with compact, deduplicated legends
    # ------------------------------------------------------------

    def _short_kernel_label(x, max_chars=20):
        s = str(x)
        return s if len(s) <= max_chars else s[: max_chars - 1] + "…"

    def _build_group_label(kernel, kind, method, include_kernel=False):
        """
        Keep legend labels compact.
        By default, legend shows only kind + method.
        Set include_kernel=True only if the number of groups is small.
        """
        if include_kernel:
            return f"{_short_kernel_label(kernel)}, {kind} / {method}"
        return f"{kind} / {method}"

    def _dedup_legend(ax, title=None, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8):
        handles, labels = ax.get_legend_handles_labels()
        seen = {}
        for h, l in zip(handles, labels):
            if l and l != "_nolegend_" and l not in seen:
                seen[l] = h

        if seen:
            ax.legend(
                seen.values(),
                seen.keys(),
                frameon=False,
                fontsize=fontsize,
                title=title,
                loc=loc,
                bbox_to_anchor=bbox_to_anchor,
                borderaxespad=0.0,
            )

    # -----------------------------
    # Figure 1: spectral bias vs spectral gap
    # -----------------------------
    fig1, ax = plt.subplots(figsize=(8.5, 4.8))

    # Set this to True only when there are very few groups
    include_kernel_in_legend = True

    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        g = g[np.isfinite(g["spectral_bias"]) & np.isfinite(g["spectral_gap"])].copy()
        if g.empty:
            continue

        label = _build_group_label(
            kernel=kernel,
            kind=kind,
            method=method,
            include_kernel=include_kernel_in_legend,
        )

        ax.scatter(
            g["spectral_bias"],
            g["spectral_gap"],
            s=20,
            alpha=0.7,
            label=label,
        )

    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Spectral gap")
    ax.set_title("Spectral bias vs Spectral gap")
    ax.grid(alpha=0.25)

    _dedup_legend(
        ax,
        title="Condition / method",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=8,
    )

    fig1.tight_layout()
    fig1.savefig(f"{out_prefix}_gap_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig1)

    # -----------------------------
    # Figure 2: spectral bias vs residual spurious score
    # -----------------------------
    fig2, ax = plt.subplots(figsize=(8.5, 4.8))

    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        g = g[np.isfinite(g["spectral_bias"]) & np.isfinite(g["residual_spurious_score"])].copy()
        if g.empty:
            continue

        label = _build_group_label(
            kernel=kernel,
            kind=kind,
            method=method,
            include_kernel=include_kernel_in_legend,
        )

        ax.scatter(
            g["spectral_bias"],
            g["residual_spurious_score"],
            s=20,
            alpha=0.7,
            label=label,
        )

    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Residual spurious score")
    ax.set_title("Spectral bias vs Residual spurious score")
    ax.grid(alpha=0.25)

    _dedup_legend(
        ax,
        title="Condition / method",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=8,
    )

    fig2.tight_layout()
    fig2.savefig(f"{out_prefix}_spurious_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig2)

    return modes_df, trials_df, summary, corr_df, fig1, fig2


In [2]:
from scipy.integrate import romb


def normalize_eigenfunctions(functions, x, density):
    abs2_eigfun = (np.abs(functions) ** 2).T  # f(x)**2
    abs2_eigfun *= density  # Compute the norm with respect to the Boltzmann measure.
    dx = x[1] - x[0]
    funcs_norm = np.sqrt(romb(abs2_eigfun, dx=dx, axis=1))  # Norms
    functions *= funcs_norm**-1.0  # Normalize
    return functions


# Function to standardise the sign of the eigenfunctions
def standardize_sign(eigenfunction, reference, x):
    eigenfunction_cut = cut_functions_to_domain(eigenfunction, x)
    reference_cut = cut_functions_to_domain(reference, x)
    norm_p = np.linalg.norm(eigenfunction_cut + reference_cut)
    norm_m = np.linalg.norm(eigenfunction_cut - reference_cut)
    if norm_p <= norm_m:
        return -1.0 * eigenfunction
    else:
        return eigenfunction


# Function to fit eigenfunctions to domain of references
def cut_functions_to_domain(functions, x, x_lims=(-1, 1)):
    mask = (x >= x_lims[0]) & (x <= x_lims[1])
    return functions[mask]


def compute_ylims(reference, margin=0.1):
    ref_min = reference.min()
    ref_max = reference.max()
    if ref_min < 0:
        ref_min *= 1 + margin
    else:
        ref_min *= 1 - margin
    if ref_max < 0:
        ref_max *= 1 - margin
    else:
        ref_max *= 1 + margin

    y_min = np.round(ref_min, 1)
    y_max = np.round(ref_max, 1)
    return y_min, y_max


def plot_eigenfunctions_with_uncertainty(results, reference_eigfuns, x):
    tab_colors = plt.get_cmap("tab10").colors
    fig, axs = plt.subplots(ncols=4, figsize=(9, 2))
    for fun_id, ax in enumerate(axs):
        reference = reference_eigfuns[:, fun_id]
        for color, (method, functions) in zip(tab_colors, results.items()):
            # Standardize signs according to the reference
            functions = np.array(functions)  # Convert list of arrays into array
            n_repetitions = functions.shape[0]
            for i in range(n_repetitions):
                for j in range(4):
                    functions[i, :, j] = standardize_sign(
                        functions[i, :, j], reference_eigfuns[:, j], x
                    )
            m = functions.mean(0)[:, fun_id].real
            st = functions.std(0)[:, fun_id].real
            ax.plot(x, m, color=color, label=method)
            ax.fill_between(x, m - st, m + st, color=color, alpha=0.3)
            ax.set_ylim(compute_ylims(reference))
        ax.plot(x, reference, color="k", lw=1, ls="--", label="Reference")
        ax.set_title(rf"Eigenfunction $\psi_{fun_id}$", fontsize=9)
        ax.set_xlim(-1, 1)
    handles, labels = ax.get_legend_handles_labels()
    ncols = 2 if (len(results) + 1) == 4 else 3
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncols=ncols,
        frameon=False,
        bbox_to_anchor=(0.0, 1.15, 1.0, 0.102),
    )


## 2. Encoders, training, and the SSL → metrics bridge

Faithful to the skeleton's `SimpleMLP`/`FeatureMap` (generalised with `input_dim` so 2-D and one-hot graph
states work). The bridge computes the induced Gram matrices K_X = ΦΦᵀ, K_YX = Φ_YΦ_Xᵀ and runs the identical
per-mode metric computations used in the kernel loops — same columns, same thresholds.

In [ ]:
import os

import torch
from torch.utils.data import DataLoader, TensorDataset

from kooplearn.linear_model import Ridge
from kooplearn.torch.nn import SpectralContrastiveLoss
# from kooplearn.torch.nn import VampLoss   # <- alternative SSL objective, same interface

device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_SSL = os.path.join("..", "analysis", "ssl")
os.makedirs(OUT_SSL, exist_ok=True)

# ---------------- experiment configuration (kept deliberately small) ----------
LATENT_DIMS = [4, 8, 16]
SEEDS = [0, 1, 2]
NUM_EPOCHS = 60
LEARNING_RATE = 1e-3
BATCH_SIZE = 512
LAYER_DIMS = [32, 64, 32]
N_TRAIN, N_VAL = 2000, 500
N_SHOW = 3  # modes analysed, matching the kernel runs
RESID_DELTA = 0.1  # spurious_residual threshold (match your kernel loops)
MODE_W = {1: 0.5, 2: 0.3, 3: 0.2}


class SimpleMLP(torch.nn.Module):
    """Skeleton MLP, generalised with input_dim (1 for Langevin, 2 for
    Duffing/linear, 12 for one-hot graph states)."""

    def __init__(self, input_dim: int, latent_dim: int, layer_dims=None, activation=torch.nn.SiLU):
        super().__init__()
        layer_dims = layer_dims or LAYER_DIMS
        lin_dims = [input_dim] + list(layer_dims) + [latent_dim]
        layers = []
        for i in range(len(lin_dims) - 2):
            layers.append(torch.nn.Linear(lin_dims[i], lin_dims[i + 1], bias=False))
            layers.append(activation())
        layers.append(torch.nn.Linear(lin_dims[-2], lin_dims[-1], bias=True))
        self.layers = torch.nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class FeatureMap(torch.nn.Module):
    """Encoder backbone + linear lagged predictor (used only during training)."""

    def __init__(self, input_dim: int, latent_dim: int, normalize_latents=False):
        super().__init__()
        self.normalize_latents = normalize_latents
        self.backbone = SimpleMLP(input_dim, latent_dim)
        self.lin = torch.nn.Linear(latent_dim, latent_dim, bias=False)

    def forward(self, X, lagged: bool = False):
        z = self.backbone(X)
        if self.normalize_latents:
            z = torch.nn.functional.normalize(z, dim=-1)
        if lagged:
            z = self.lin(z)
        return z


def train_encoder(
    X_train,
    X_val,
    input_dim,
    latent_dim,
    seed,
    criterion=None,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
):
    """Train one encoder; returns (model, diagnostics dict with loss curves)."""
    criterion = criterion or SpectralContrastiveLoss()
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = FeatureMap(input_dim, latent_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    tr = torch.from_numpy(np.asarray(X_train, np.float32))
    va = torch.from_numpy(np.asarray(X_val, np.float32))
    train_dl = DataLoader(TensorDataset(tr[:-1], tr[1:]), batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(
        TensorDataset(va[:-1], va[1:]), batch_size=max(len(va) - 1, 1), shuffle=False
    )

    def step(batch, is_train=True):
        bX, bY = (b.to(device) for b in batch)
        if is_train:
            optimizer.zero_grad()
        loss = criterion(model(bX), model(bY, lagged=True))
        if is_train:
            loss.backward()
            optimizer.step()
        return loss.item()

    curve_tr, curve_va = [], []
    for _ in range(num_epochs):
        model.train()
        curve_tr.append(np.mean([step(b) for b in train_dl]))
        model.eval()
        with torch.no_grad():
            curve_va.append(np.mean([step(b, False) for b in val_dl]))

    return model, {
        "loss_curve_train": curve_tr,
        "loss_curve_val": curve_va,
        "final_train_loss": curve_tr[-1],
        "final_val_loss": curve_va[-1],
    }


def embed(model, X):
    """Latent features of the (un-lagged) encoder, as numpy."""
    model.eval()
    with torch.no_grad():
        z = model(torch.from_numpy(np.asarray(X, np.float32)).to(device))
    return z.cpu().numpy().astype(np.float64)


def effective_rank(M, tol=1e-8):
    s = np.linalg.svd(np.asarray(M), compute_uv=False)
    return int(np.sum(s > tol * s.max())) if s.size and s.max() > 0 else 0


In [12]:
def _sym_pow(M, p, eps=1e-10):
    """Symmetric PSD matrix power with eigenvalue clipping (for CCA whitening)."""
    w, V = np.linalg.eigh(np.asarray(M, float))
    w = np.clip(w, eps, None)
    return (V * w**p) @ V.T


def cca_readout(Phi_X, Phi_Y, r):
    """CCA / VAMP readout: two-sided whitening. Truncation proxy is
    sigma_{r+1} of the whitened operator A = Cx^{-1/2} Cxy Cy^{-1/2} —
    the direct analogue of pcr_truncation (sigma_{r+1}(C)) and
    rrr_truncation (sigma_{r+1}(C^{-1/2} T)).

    Returns (eigvals, latent-space eigvecs, rho_{r+1}) of the rank-r
    CCA-truncated EDMD matrix K = Cx^{-1/2} A_r Cy^{1/2}."""
    n = Phi_X.shape[0]
    Cx = Phi_X.T @ Phi_X / n
    Cy = Phi_Y.T @ Phi_Y / n
    Cxy = Phi_X.T @ Phi_Y / n
    Cx_ih = _sym_pow(Cx, -0.5)
    Cy_ih = _sym_pow(Cy, -0.5)
    Cy_h = _sym_pow(Cy, 0.5)
    A = Cx_ih @ Cxy @ Cy_ih
    U, S, Vt = np.linalg.svd(A)
    A_r = U[:, :r] @ np.diag(S[:r]) @ Vt[:r]
    K = Cx_ih @ A_r @ Cy_h
    vals, vecs = np.linalg.eig(K)
    order = np.argsort(-np.abs(vals))
    vals, vecs = vals[order][:r], vecs[:, order][:, :r]
    rho = float(S[r]) if len(S) > r else 0.0
    return vals, vecs, rho


def analyse_encoder(
    model,
    X_train,
    X_val,
    Y_val,
    *,
    kind,
    kernel_label,
    trial,
    vals_ref=None,
    ref_delta=0.05,
    n_show=N_SHOW,
):
    """The SSL -> metrics bridge. Treats the learned phi exactly like a kernel:
    induced Gram matrices + the verbatim metric functions from cell 1.

    Three readouts of the SAME encoder: PCR (no whitening), RRR (one-sided),
    CCA/VAMP (two-sided — matched to what SpectralContrastiveLoss optimises,
    which is the operator's singular subspace). Returns records in the exact
    schema of the kernel experiment loops.
    """
    Phi_tr = embed(model, X_train)  # (n, d)
    Phi_Xv, Phi_Yv = embed(model, X_val), embed(model, Y_val)
    n = Phi_tr.shape[0] - 1
    Phi_X, Phi_Y = Phi_tr[:-1], Phi_tr[1:]
    K_X = Phi_X @ Phi_X.T  # induced Gram
    K_YX = Phi_Y @ Phi_X.T
    C, T = K_X / n, K_YX / n

    modes_out, trials_out, extras_out = [], [], []
    for method in ["SSL+Ridge (PCR)", "SSL+Ridge (RRR)", "SSL+CCA"]:
        if method == "SSL+CCA":
            vals_hat, vecs, rho = cca_readout(Phi_X, Phi_Y, n_show)
            funcs_hat = Phi_X @ vecs
            psi_X_val = Phi_Xv @ vecs
            psi_Y_val = Phi_Yv @ vecs
            op_rank_obj = int(np.sum(np.abs(vals_hat) > 1e-10))
        else:
            reduced_rank = method == "SSL+Ridge (RRR)"
            ridge = Ridge(n_components=n_show, reduced_rank=reduced_rank, alpha=1e-6).fit(Phi_tr)
            vals_hat, funcs_hat = ridge.eig(eval_right_on=Phi_X)
            sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
            vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]
            _, psi_X_val = ridge.eig(eval_right_on=Phi_Xv)
            _, psi_Y_val = ridge.eig(eval_right_on=Phi_Yv)
            psi_X_val = psi_X_val[:, sort_perm]
            psi_Y_val = psi_Y_val[:, sort_perm]
            if reduced_rank:
                rho = rrr_truncation(C, T, n_show)
            else:
                rho = pcr_truncation(C, n_show)
            op_rank_obj = getattr(ridge, "rank_", getattr(ridge, "rank", np.nan))

        gap = spectral_gap(vals_hat)
        rho = float(rho)
        n_spur_res, resid_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=RESID_DELTA
        )
        if vals_ref is not None:
            n_spur_ref = spurious_ref(np.asarray(vals_hat), np.asarray(vals_ref), delta=ref_delta)
        else:
            n_spur_ref = np.nan

        for j in range(min(n_show, funcs_hat.shape[1])):
            bias_j, eta_j = spectral_bias(funcs_hat[:, j], C, rho)
            modes_out.append(
                {
                    "kind": kind,
                    "kernel": kernel_label,
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(np.real(bias_j)),
                    "metric_distortion": float(np.real(eta_j)),
                    "truncation": float(rho),
                    "residual_spurious_score": float(resid_scores[j]),
                    "spectral_gap": gap,
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                }
            )
        op_rank = op_rank_obj
        trials_out.append(
            {
                "kind": kind,
                "kernel": kernel_label,
                "method": method,
                "trial": trial,
                "data_seed": trial,
                "spurious_ref_count": n_spur_ref,
                "spurious_residual_count": int(n_spur_res),
                "spectral_gap": gap,
                "rank": op_rank,
            }
        )
        extras_out.append(
            {
                "kind": kind,
                "kernel": kernel_label,
                "method": method,
                "trial": trial,
                "latent_norm_mean": float(np.linalg.norm(Phi_tr, axis=1).mean()),
                "latent_norm_std": float(np.linalg.norm(Phi_tr, axis=1).std()),
                "latent_cov_rank": effective_rank(Phi_tr.T @ Phi_tr / len(Phi_tr)),
                "operator_rank": op_rank,
            }
        )
    return modes_out, trials_out, extras_out


def run_ssl_system(system, data_fn, kinds, input_dim, vals_ref_fn=None, ref_delta=0.05):
    """Full sweep for one system: kinds x latent dims x seeds."""
    modes, trials, extras, train_diag = [], [], [], []
    for kind_val in kinds:
        kind = str(kind_val[0]) if isinstance(kind_val, tuple) else str(kind_val)
        vals_ref = vals_ref_fn(kind_val) if vals_ref_fn is not None else None
        for d in LATENT_DIMS:
            for seed in SEEDS:
                X_train, X_val, Y_val = data_fn(kind_val, seed)
                model, diag = train_encoder(X_train, X_val, input_dim, d, seed)
                label = f"encoder(d={d})"
                m, t, e = analyse_encoder(
                    model,
                    X_train,
                    X_val,
                    Y_val,
                    kind=kind,
                    kernel_label=label,
                    trial=seed,
                    vals_ref=vals_ref,
                    ref_delta=ref_delta,
                )
                modes += m
                trials += t
                extras += e
                train_diag.append(
                    {"system": system, "kind": kind, "kernel": label, "trial": seed} | diag
                )
                print(
                    f"[{system}] {kind} d={d} seed={seed} "
                    f"loss {diag['final_train_loss']:.3f}/{diag['final_val_loss']:.3f}"
                )
    prefix = os.path.join(OUT_SSL, f"{system}_ssl")
    analyse_spectrum(modes, trials, prefix)  # her pipeline, unchanged
    td = pd.DataFrame(train_diag)
    td_flat = td.drop(columns=["loss_curve_train", "loss_curve_val"])
    td_flat = td_flat.merge(pd.DataFrame(extras), on=["kind", "kernel", "trial"], how="left")
    td_flat.to_csv(f"{prefix}_training_diagnostics.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 4))
    for _, row in td.iterrows():
        ax.plot(row["loss_curve_train"], alpha=0.5, lw=0.8)
    ax.set_xlabel("epoch")
    ax.set_ylabel("train loss")
    ax.set_title(f"{system}: SSL training curves (all configs)")
    fig.tight_layout()
    fig.savefig(f"{prefix}_loss_curves.png", dpi=200)
    plt.close(fig)
    return td_flat


## 3. System runs

### 3a. Langevin (Prinz potential) — reversible baseline

In [13]:
GAMMA_LG, SIGMA_LG, DT_LG, SUB_LG = 1.0, 2.0, 1e-4, 100
N_BURN = 200

prinz_ref = compute_prinz_potential_eig(GAMMA_LG, SIGMA_LG, DT_LG, num_components=5)
if isinstance(prinz_ref, tuple):
    prinz_ref = prinz_ref[0]
prinz_ref = np.atleast_1d(np.asarray(prinz_ref, dtype=complex))


def data_langevin(kind_val, seed):
    n_steps = (N_TRAIN + N_VAL + N_BURN) * SUB_LG
    data = make_prinz_potential(
        X0=0, n_steps=n_steps, gamma=GAMMA_LG, sigma=SIGMA_LG, random_state=seed
    )
    arr = data.values[::SUB_LG][N_BURN:]
    X_train = arr[:N_TRAIN]
    val = arr[N_TRAIN : N_TRAIN + N_VAL]
    return X_train, val[:-1], val[1:]


diag_lg = run_ssl_system(
    "langevin",
    data_langevin,
    kinds=["all"],
    input_dim=1,
    vals_ref_fn=lambda k: prinz_ref,
    ref_delta=0.05,
)


[langevin] all d=4 seed=0 loss -0.896/-0.880
[langevin] all d=4 seed=1 loss -1.831/-1.781
[langevin] all d=4 seed=2 loss -1.785/-1.018
[langevin] all d=8 seed=0 loss -1.835/-1.577
[langevin] all d=8 seed=1 loss -1.866/-1.824
[langevin] all d=8 seed=2 loss -1.867/-1.301
[langevin] all d=16 seed=0 loss -1.871/-1.677
[langevin] all d=16 seed=1 loss -1.883/-1.862
[langevin] all d=16 seed=2 loss -1.907/-1.409


### 3b. Duffing — strongly non-normal (RQ2)

In [14]:
from kooplearn.datasets import make_duffing

DT_DUF, SUB_DUF = 1e-3, 20
OMEGAS_DUF = [0.5, 1.5, 3.0]


def data_duffing(kind_val, seed):
    omega = float(str(kind_val).split("=")[1])

    def sim(n_keep, rs):
        rng = np.random.default_rng(rs)
        X0 = rng.uniform(-0.5, 0.5, size=2)
        df = make_duffing(
            X0=X0,
            n_steps=n_keep * SUB_DUF,
            dt=DT_DUF,
            alpha=-1.0,
            beta=1.0,
            gamma=0.3,
            delta=0.2,
            omega=omega,
        )
        return df.values[::SUB_DUF][:n_keep]

    X_train = sim(N_TRAIN, 10_000 + seed)
    val = sim(N_VAL, 20_000 + seed)
    return X_train, val[:-1], val[1:]


diag_duf = run_ssl_system(
    "duffing", data_duffing, kinds=[f"omega={w}" for w in OMEGAS_DUF], input_dim=2, vals_ref_fn=None
)


[duffing] omega=0.5 d=4 seed=0 loss -1.001/-1.003
[duffing] omega=0.5 d=4 seed=1 loss -2.008/-1.318
[duffing] omega=0.5 d=4 seed=2 loss -1.001/-0.829
[duffing] omega=0.5 d=8 seed=0 loss -1.985/9.474
[duffing] omega=0.5 d=8 seed=1 loss -2.015/-1.179
[duffing] omega=0.5 d=8 seed=2 loss -2.018/4.645
[duffing] omega=0.5 d=16 seed=0 loss -2.096/7.187
[duffing] omega=0.5 d=16 seed=1 loss -2.351/-1.787
[duffing] omega=0.5 d=16 seed=2 loss -2.001/4.198
[duffing] omega=1.5 d=4 seed=0 loss -1.001/-0.774
[duffing] omega=1.5 d=4 seed=1 loss -1.000/-0.656
[duffing] omega=1.5 d=4 seed=2 loss -1.002/-0.985
[duffing] omega=1.5 d=8 seed=0 loss -2.089/59.788
[duffing] omega=1.5 d=8 seed=1 loss -1.000/-0.247
[duffing] omega=1.5 d=8 seed=2 loss -2.019/-0.987
[duffing] omega=1.5 d=16 seed=0 loss -3.001/53.130
[duffing] omega=1.5 d=16 seed=1 loss -1.033/-0.686
[duffing] omega=1.5 d=16 seed=2 loss -2.187/-1.283
[duffing] omega=3.0 d=4 seed=0 loss -1.000/-0.624
[duffing] omega=3.0 d=4 seed=1 loss -0.999/-0.63

### 3c. Linear system — analytic control (RQ3)

In [15]:
from kooplearn.datasets import make_linear_system

DT_LIN, SUB_LIN, BETA_LIN = 1e-3, 20, 1.0
A_CASES = {"A=rev": np.diag([-1.0, -2.0]), "A=rot": np.array([[-1.0, 2.0], [-2.0, -1.5]])}
LAG_LIN = DT_LIN * SUB_LIN


def linear_ref(kind):
    A = A_CASES[str(kind)]
    M = expm(A * LAG_LIN)
    lam = np.linalg.eigvals(M)
    ref = [1.0 + 0j] + [complex(z) for z in lam]
    for i in range(len(lam)):
        for j in range(i, len(lam)):
            ref.append(complex(lam[i] * lam[j]))
    return np.array(ref[:5])


def data_linear(kind_val, seed):
    A = A_CASES[str(kind_val)]
    d = A.shape[0]
    A_disc = np.eye(d) + DT_LIN * A
    noise = np.sqrt(2.0 * DT_LIN / BETA_LIN)

    def sim(n_keep, rs):
        df = make_linear_system(
            X0=np.zeros(d), A=A_disc, n_steps=n_keep * SUB_LIN, noise=noise, random_state=rs
        )
        return df.values[::SUB_LIN][:n_keep]

    X_train = sim(N_TRAIN, 1000 + seed)
    val = sim(N_VAL, 9000 + seed)
    return X_train, val[:-1], val[1:]


diag_lin = run_ssl_system(
    "linear", data_linear, kinds=list(A_CASES), input_dim=2, vals_ref_fn=linear_ref, ref_delta=0.1
)


[linear] A=rev d=4 seed=0 loss -0.953/-0.545
[linear] A=rev d=4 seed=1 loss -1.925/-1.253
[linear] A=rev d=4 seed=2 loss -1.929/-1.221
[linear] A=rev d=8 seed=0 loss -2.829/-1.727
[linear] A=rev d=8 seed=1 loss -1.935/-1.280
[linear] A=rev d=8 seed=2 loss -2.015/-1.275
[linear] A=rev d=16 seed=0 loss -3.016/-2.109
[linear] A=rev d=16 seed=1 loss -2.865/-1.884
[linear] A=rev d=16 seed=2 loss -2.909/-2.032
[linear] A=rot d=4 seed=0 loss -0.961/-0.932
[linear] A=rot d=4 seed=1 loss -1.876/-1.638
[linear] A=rot d=4 seed=2 loss -1.838/-1.387
[linear] A=rot d=8 seed=0 loss -2.857/-2.460
[linear] A=rot d=8 seed=1 loss -2.854/-2.383
[linear] A=rot d=8 seed=2 loss -1.969/-1.578
[linear] A=rot d=16 seed=0 loss -3.424/-3.007
[linear] A=rot d=16 seed=1 loss -2.906/-2.342
[linear] A=rot d=16 seed=2 loss -2.923/-2.399


### 3d. Graph random walks — cycle-12 and path-12 (RQ4), one-hot inputs

In [16]:
N_NODES = 12


def cycle_P(n=N_NODES):
    P = np.zeros((n, n))
    for i in range(n):
        P[i, (i - 1) % n] = P[i, (i + 1) % n] = 0.5
    return P


def path_P(n=N_NODES):
    P = np.zeros((n, n))
    for i in range(n):
        nbrs = [j for j in (i - 1, i + 1) if 0 <= j < n]
        for j in nbrs:
            P[i, j] = 1.0 / len(nbrs)
    return P


GRAPHS = {"cycle_12": cycle_P(), "path_12": path_P()}


def graph_ref(kind):
    v = np.linalg.eigvals(GRAPHS[str(kind)])
    return v[np.argsort(-np.abs(v))][:5]


def simulate_markov_chain(P, n_steps, random_state, x0=0):
    rng = np.random.default_rng(random_state)
    states = np.empty(n_steps, dtype=int)
    states[0] = x0
    for t in range(n_steps - 1):
        states[t + 1] = rng.choice(P.shape[0], p=P[states[t]])
    return states


def data_graph(kind_val, seed):
    P = GRAPHS[str(kind_val)]
    onehot = np.eye(N_NODES)
    tr = onehot[simulate_markov_chain(P, N_TRAIN, 30_000 + seed)]
    va = onehot[simulate_markov_chain(P, N_VAL, 40_000 + seed)]
    return tr, va[:-1], va[1:]


diag_gr = run_ssl_system(
    "graph",
    data_graph,
    kinds=list(GRAPHS),
    input_dim=N_NODES,
    vals_ref_fn=graph_ref,
    ref_delta=0.15,
)


[graph] cycle_12 d=4 seed=0 loss -1.000/-1.000
[graph] cycle_12 d=4 seed=1 loss -1.002/-1.003
[graph] cycle_12 d=4 seed=2 loss -1.002/-1.003
[graph] cycle_12 d=8 seed=0 loss -1.000/-1.000
[graph] cycle_12 d=8 seed=1 loss -1.000/-1.000
[graph] cycle_12 d=8 seed=2 loss -1.000/-1.000
[graph] cycle_12 d=16 seed=0 loss -1.000/-1.000
[graph] cycle_12 d=16 seed=1 loss -1.000/-1.000
[graph] cycle_12 d=16 seed=2 loss -1.000/-1.000
[graph] path_12 d=4 seed=0 loss -1.000/-1.000
[graph] path_12 d=4 seed=1 loss -1.162/-1.204
[graph] path_12 d=4 seed=2 loss -1.001/-1.001
[graph] path_12 d=8 seed=0 loss -1.000/-1.000
[graph] path_12 d=8 seed=1 loss -1.054/-1.066
[graph] path_12 d=8 seed=2 loss -1.000/-0.999
[graph] path_12 d=16 seed=0 loss -1.000/-1.000
[graph] path_12 d=16 seed=1 loss -1.000/-1.000
[graph] path_12 d=16 seed=2 loss -1.000/-1.000


## 4. Unified scoring: learned encoders vs analytic kernels

The `analyse_spectrum` outputs above have the same schema as `benchmarks/`, so the frozen unified weights
(baseline + horizon terms) apply unchanged. Encoders are scored (a) among themselves per system and (b)
**pooled with every analytic kernel** from `unified_scoring_database.csv` per (system, kind) — the
learned-vs-analytic head-to-head. Per the practical cautions: z-scoring is pool-relative, so if encoder metric
distributions shift the normalisation, that shows up here transparently (the raw metric columns are kept
alongside the scores).

In [17]:
from scipy.stats import spearmanr

UNIFIED_DB = os.path.join("..", "analysis", "unified_scoring_database.csv")
H = 50
BASE_WEIGHTS = {
    "agg_bias_mean": 1.0,
    "agg_dist_mean": 1.0,
    "agg_spurious_mean": 1.0,
    "agg_trunc_mean": 0.5,
    "agg_bias_std": 0.25,
    "agg_spurious_std": 0.25,
    "mean_spurious_ref_count": 0.5,
    "mean_spurious_residual_count": 1.0,
    "mean_spectral_gap": 0.5,
}
UNIFIED_WEIGHTS = BASE_WEIGHTS | {"agg_horizon_instab": 1.0, "agg_horizon_sens": 1.0}
LARGER_IS_BETTER = {"mean_spectral_gap"}


def _zscore(x, flip=False):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    if flip:
        x = -x
    x = x.replace([np.inf, -np.inf], np.nan)
    sd = x.std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(0.0, index=x.index)
    return ((x - x.mean()) / sd).fillna(0.0)


def score_pool(df, group_within, score_name="score_unified_ssl", weights=UNIFIED_WEIGHTS):
    df = df.copy()
    used = [m for m in weights if m in df.columns and df[m].notna().any()]
    for m in used:
        df[m + "_z"] = df.groupby(list(group_within), dropna=False)[m].transform(
            lambda s: _zscore(s, m in LARGER_IS_BETTER)
        )
    df[score_name] = sum(weights[m] * df[m + "_z"] for m in used)
    df["rank_" + score_name] = (
        df.groupby(list(group_within), dropna=False)[score_name].rank(method="first").astype(int)
    )
    return df


def aggregates_from_ssl(system):
    """kernel_spectral_score-style aggregates + horizon terms from the
    analyse_spectrum outputs written in section 3."""
    prefix = os.path.join(OUT_SSL, f"{system}_ssl")
    summ = pd.read_csv(f"{prefix}_summary.csv")
    met = pd.read_csv(f"{prefix}_metrics.csv")
    tri = pd.read_csv(f"{prefix}_trials.csv")

    summ = summ[summ["eigenfunction_id"].isin(MODE_W)].copy()
    summ["w"] = summ["eigenfunction_id"].map(MODE_W)
    key = ["kernel", "kind", "method"]

    def wavg(col):
        g = summ.assign(v=summ["w"] * summ[col]).groupby(key)[["v", "w"]].sum()
        return g["v"] / g["w"]

    agg = pd.DataFrame(
        {
            "agg_bias_mean": wavg("bias_mean"),
            "agg_bias_std": wavg("bias_std"),
            "agg_dist_mean": wavg("dist_mean"),
            "agg_trunc_mean": wavg("trunc_mean"),
            "agg_spurious_mean": wavg("spurious_mean"),
            "agg_spurious_std": wavg("spurious_std"),
        }
    )
    tri_agg = tri.groupby(key).agg(
        mean_spurious_ref_count=("spurious_ref_count", "mean"),
        mean_spurious_residual_count=("spurious_residual_count", "mean"),
        mean_spectral_gap=("spectral_gap", "mean"),
    )
    agg = agg.join(tri_agg)

    # horizon terms from est_eig (same definitions as the horizon notebook)
    met = met[met["eigenfunction_id"].isin(MODE_W)].copy()
    met["est"] = met["est_eig_real"] + 1j * met["est_eig_imag"]
    t = np.arange(1, H + 1)
    hrows = {}
    for k, g in met.groupby(key):
        instab = sens = wsum = 0.0
        for mode, gm in g.groupby("eigenfunction_id"):
            w = MODE_W[mode]
            lam = gm["est"].mean()
            sig = (
                np.sqrt(gm["est_eig_real"].std() ** 2 + gm["est_eig_imag"].std() ** 2)
                if len(gm) > 1
                else 0.0
            )
            instab += w * max(0.0, abs(lam) - 1.0)
            sens += w * np.mean(t * np.abs(lam) ** (t - 1) * sig)
            wsum += w
        hrows[k] = {"agg_horizon_instab": instab / wsum, "agg_horizon_sens": sens / wsum}
    agg = agg.join(pd.DataFrame(hrows).T.rename_axis(key))
    agg = agg.reset_index()
    agg["system"], agg["family"] = system, "ssl_encoder"
    return agg


ssl_agg = pd.concat(
    [aggregates_from_ssl(s) for s in ["langevin", "duffing", "linear", "graph"]], ignore_index=True
)

# (a) encoders scored among themselves, per system/kind
ssl_scored = score_pool(ssl_agg, ("system", "kind"), "score_ssl_only")
ssl_scored.to_csv(os.path.join(OUT_SSL, "ssl_scores.csv"), index=False)

# (b) pooled with every analytic kernel from the unified database
uni = pd.read_csv(UNIFIED_DB)
uni_metrics = uni[
    ["family", "system", "kind", "kernel", "method"]
    + [c for c in UNIFIED_WEIGHTS if c in uni.columns]
]


# align kinds: numeric kinds in the analytic DB (rbf ou/langevin sweeps) -> 'all'
def norm_kind(k):
    try:
        float(str(k))
        return "all"
    except ValueError:
        return str(k)


uni_metrics = uni_metrics.assign(kind=uni_metrics["kind"].map(norm_kind))
pool = pd.concat([uni_metrics, ssl_agg], ignore_index=True)
pool = pool[
    pool["system"].isin(ssl_agg["system"].unique()) & pool["kind"].isin(ssl_agg["kind"].unique())
]
pool = score_pool(pool, ("system", "kind"), "score_vs_analytic")
pool.to_csv(os.path.join(OUT_SSL, "ssl_vs_analytic_pooled.csv"), index=False)

print("Best learned-encoder rank among all candidates, per (system, kind):")
best_enc = (
    pool[pool["family"] == "ssl_encoder"]
    .groupby(["system", "kind"])["rank_score_vs_analytic"]
    .min()
)
n_cand = pool.groupby(["system", "kind"])["kernel"].count()
print(pd.DataFrame({"best_encoder_rank": best_enc, "n_candidates": n_cand}).to_string())


Best learned-encoder rank among all candidates, per (system, kind):
                    best_encoder_rank  n_candidates
system   kind                                      
duffing  omega=0.5                 11           125
         omega=1.5                  3           125
         omega=3.0                  3           125
graph    cycle_12                   1            57
         path_12                    4            57
langevin all                        1           125
linear   A=rev                      2           125
         A=rot                      1           125


## 5. Hypothesis tests

In [18]:
# ---- H1: on the reversible system, do encoders rank with the best analytic kernels?
lg = pool[pool["system"] == "langevin"].sort_values("score_vs_analytic")
print("H1 — Langevin, top 10 of the pooled ranking:")
print(
    lg[
        [
            "family",
            "kernel",
            "method",
            "score_vs_analytic",
            "agg_bias_mean",
            "agg_dist_mean",
            "mean_spurious_residual_count",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

# ---- H2: non-normal regime — spurious spectrum, encoders vs analytic
duf = pool[pool["system"] == "duffing"]
h2 = (
    duf.assign(is_encoder=duf["family"] == "ssl_encoder")
    .groupby(["kind", "is_encoder"])["mean_spurious_residual_count"]
    .median()
    .unstack()
)
h2.columns = ["analytic", "encoder"]
print("\nH2 — Duffing: median spurious-residual count (lower = better):")
print(h2.round(2).to_string())

# ---- H3: does small latent dim show up as high distortion (and does the score see it)?
ssl_scored["d"] = ssl_scored["kernel"].str.extract(r"d=(\d+)").astype(int)
h3 = ssl_scored.groupby(["system", "d"])[["agg_dist_mean", "score_ssl_only"]].median().round(3)
print("\nH3 — distortion and score vs latent dim:")
print(h3.to_string())
for s, g in ssl_scored.groupby("system"):
    if g["agg_dist_mean"].nunique() > 2:
        rho = spearmanr(g["agg_dist_mean"], g["score_ssl_only"]).statistic
        print(
            f"  {s}: Spearman(distortion, score) = {rho:.2f} "
            "(positive = score penalises mismatch, supporting H3)"
        )


H1 — Langevin, top 10 of the pooled ranking:
     family        kernel          method  score_vs_analytic  agg_bias_mean  agg_dist_mean  mean_spurious_residual_count
ssl_encoder encoder(d=16) SSL+Ridge (PCR)          -6.040359   1.222812e-03      11.399825                      2.000000
ssl_encoder encoder(d=16) SSL+Ridge (RRR)          -6.017876   7.438806e-01      23.521290                      2.000000
       poly     poly(p=1)    Reduced Rank          -5.360165   1.198679e-15       0.486631                      1.400000
ssl_encoder  encoder(d=8) SSL+Ridge (RRR)          -5.343623   4.902896e-01      26.300971                      2.333333
ssl_encoder  encoder(d=8) SSL+Ridge (PCR)          -5.316521   1.112136e-03      14.155051                      2.333333
       poly     poly(p=4)    Reduced Rank          -4.359994   2.538960e-14       7.279239                      2.100000
ssl_encoder  encoder(d=4) SSL+Ridge (PCR)          -2.633689   8.113938e-05      36.565526                  

## 6. Outputs & interpretation rules

All in `analysis/ssl/`: per system `*_ssl_{summary,metrics,trials,corr}.csv` (schema-identical to
`benchmarks/`, so every existing analysis notebook can consume them), `*_ssl_training_diagnostics.csv`
(loss curves, latent norms, covariance/operator ranks — check `latent_cov_rank` ≪ d for collapse),
`*_ssl_loss_curves.png`, `ssl_scores.csv`, and `ssl_vs_analytic_pooled.csv`.

Reading the results (agreed rules): encoders are treated exactly like kernels under J(φ) = the unified
composite with frozen weights. If pooled top picks also have low bias/horizon error → the score generalises
to learned operators (H1). If encoders beat analytic kernels on spurious counts at comparable bias in the
Duffing regimes → major positive result; if they fail there too → confirms the remaining limitation and
motivates encoder + spectral-regularizer combinations (H2). If d=4 rows show inflated η̂ that the score
penalises → the score detects representational mismatch (H3). If encoder metric distributions clearly shift
the pooled z-normalisation (e.g. one encoder outlier compresses all analytic z-scores), report re-calibrated
weights transparently — the raw metric columns in both CSVs are retained for exactly that check.